# 02 — Comparación de modelos

Esta notebook resume la comparación del baseline, Ridge sobre `log(precio)` y LightGBM cuantílico para recomendar un precio nocturno a anfitriones nuevos de Airbnb en Ciudad de México.

La notebook es deliberadamente un **lector de resultados**: no entrena modelos, no descarga datos y no abre ni evalúa el test temporal. Solo usa resultados de desarrollo que hayan sido generados por el ejecutor común y guardados en las rutas documentadas.

## Pregunta y modelos comparados

La decisión práctica es proponer un precio por noche en MXN para un anuncio nuevo, junto con un rango de referencia. La comparación debe usar los mismos predictores, las mismas particiones agrupadas por `host_id` y las mismas métricas.

- **Baseline:** cuantiles empíricos por alcaldía y tipo de alojamiento, con respaldos para grupos pequeños o categorías no observadas.
- **Ridge:** regresión regularizada sobre `log(precio)`, con preprocesamiento aprendido dentro de cada fold y regreso a MXN para predecir.
- **LightGBM cuantílico:** modelo de árboles que debe producir los cuatro cuantiles del contrato común.

La métrica de selección es el pinball loss de p35. También se observan el MAE del precio recomendado p35 y la cobertura del intervalo p10–p90.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def localizar_raiz_repositorio():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    for candidato in candidatos:
        if (candidato / 'README.md').exists() and (candidato / 'docs').exists():
            return candidato
    return Path.cwd()


REPO_ROOT = localizar_raiz_repositorio()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RESULTS_PATH = REPO_ROOT / 'docs' / 'model_comparison.json'
PREDICTIONS_PATH = REPO_ROOT / 'data' / 'processed' / 'validation_predictions.csv'
SEGMENT_CSV_PATH = REPO_ROOT / 'data' / 'processed' / 'segment_tables.csv'
SEGMENT_MD_PATH = REPO_ROOT / 'docs' / 'segment_tables.md'

print(f'Raíz detectada: {REPO_ROOT}')
print(f'Resultados comunes: {RESULTS_PATH.relative_to(REPO_ROOT)}')
print(f'Predicciones de validación: {PREDICTIONS_PATH.relative_to(REPO_ROOT)}')
print(f'Tablas estructuradas por segmento: {SEGMENT_CSV_PATH.relative_to(REPO_ROOT)}')
print(f'Documento legible por segmento: {SEGMENT_MD_PATH.relative_to(REPO_ROOT)}')

In [ ]:
# Se intenta usar el contrato implementado; el respaldo permite explicar la falta de integración
# sin producir un traceback si esta notebook se abre antes de incorporar todos los módulos.
try:
    from src.metrics import NOMINAL_COVERAGE, QUANTILE_LEVELS, SELECTION_METRIC
    FUENTE_CONTRATO = 'src.metrics.py'
except (ImportError, ModuleNotFoundError):
    QUANTILE_LEVELS = {'p10': 0.10, 'p35': 0.35, 'p50': 0.50, 'p90': 0.90}
    NOMINAL_COVERAGE = 0.80
    SELECTION_METRIC = 'pinball_recommended'
    FUENTE_CONTRATO = 'respaldo documentado; falta src.metrics.py en esta copia'

print(f'Contrato de métricas usado: {FUENTE_CONTRATO}')
print(f'Cuantiles esperados: {list(QUANTILE_LEVELS)}')
print(f'Métrica de selección: {SELECTION_METRIC}')
print(f'Cobertura nominal p10-p90: {NOMINAL_COVERAGE:.0%}')

## Qué significan los cuantiles

p10, p35, p50 y p90 son cuantiles de la **distribución estimada del precio publicado**, condicionada a las características del anuncio. p35 es la recomendación operativa y p10–p90 es un rango de referencia.

Estos cuantiles no son intervalos de confianza sobre la media ni probabilidades de reserva. Tampoco garantizan el precio que se cobrará, la ocupación o los ingresos. La cobertura nominal del rango p10–p90 es 80 %, pero la cobertura observada se debe medir en datos de validación y no se garantiza para cada anuncio.

In [ ]:
def cargar_y_validar_comparacion(path):
    """Carga únicamente un registro de desarrollo con el esquema del ejecutor común."""
    if not path.exists():
        return None, [f'No existe {path.relative_to(REPO_ROOT)}.']
    try:
        contenido = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, UnicodeError, json.JSONDecodeError) as error:
        return None, [f'No se pudo leer el JSON de resultados: {error}.']

    problemas = []
    if not isinstance(contenido, dict):
        return None, ['El archivo de comparación no contiene un objeto JSON.']
    if contenido.get('snapshot') != '2026-06-15':
        problemas.append("El registro no identifica el snapshot de desarrollo 2026-06-15.")
    if contenido.get('test_evaluated') is not False:
        problemas.append('El registro no confirma explícitamente que el test no fue evaluado.')
    if contenido.get('selection_metric') not in (None, SELECTION_METRIC):
        problemas.append(f"La métrica de selección declarada no coincide con {SELECTION_METRIC}.")

    cuantiles_archivo = contenido.get('quantiles')
    if not isinstance(cuantiles_archivo, dict):
        problemas.append('Falta la clave quantiles o no contiene un objeto.')
    else:
        faltantes = sorted(set(QUANTILE_LEVELS) - set(cuantiles_archivo))
        extras = sorted(set(cuantiles_archivo) - set(QUANTILE_LEVELS))
        if faltantes or extras:
            problemas.append(
                f'La clave quantiles no coincide con el contrato: faltantes={faltantes}, extras={extras}.'
            )
        for nombre, esperado in QUANTILE_LEVELS.items():
            if nombre not in cuantiles_archivo:
                continue
            try:
                observado = float(cuantiles_archivo[nombre])
            except (TypeError, ValueError):
                problemas.append(f'El cuantil {nombre} no es numérico; se esperaba {esperado:.2f}.')
                continue
            if not np.isfinite(observado) or not np.isclose(observado, esperado, rtol=0.0, atol=1e-12):
                problemas.append(
                    f'El cuantil {nombre} no coincide con el contrato: '
                    f'se observó {observado!r} y se esperaba {esperado:.2f}.'
                )

    modelos = contenido.get('models')
    if not isinstance(modelos, list) or not modelos:
        problemas.append('No hay una lista no vacía de modelos.')
        return None, problemas

    filas = []
    campos = ('pinball_recommended', 'mae_recommended', 'coverage')
    for registro in modelos:
        nombre = registro.get('model') if isinstance(registro, dict) else None
        medias = registro.get('mean', {}) if isinstance(registro, dict) else {}
        if not nombre or not isinstance(medias, dict):
            problemas.append('Hay un registro de modelo sin nombre o sin resumen mean.')
            continue
        fila = {'modelo': str(nombre)}
        fila_valida = True
        for campo in campos:
            try:
                valor = float(medias[campo])
            except (KeyError, TypeError, ValueError):
                problemas.append(f'Falta {campo} para el modelo {nombre}.')
                fila_valida = False
                continue
            if not np.isfinite(valor):
                problemas.append(f'{campo} no es finito para el modelo {nombre}.')
                fila_valida = False
            fila[campo] = valor
        if fila_valida:
            if fila['pinball_recommended'] < 0 or fila['mae_recommended'] < 0:
                problemas.append(f'Hay una métrica negativa en el modelo {nombre}.')
                fila_valida = False
            if not 0 <= fila['coverage'] <= 1:
                problemas.append(f'La cobertura está fuera de [0, 1] en el modelo {nombre}.')
                fila_valida = False
        if fila_valida:
            filas.append(fila)

    if problemas or not filas:
        if not filas and not problemas:
            problemas.append('No quedaron filas válidas para comparar.')
        return None, problemas
    return {'registro': contenido, 'tabla': pd.DataFrame(filas)}, []


comparacion, problemas_comparacion = cargar_y_validar_comparacion(RESULTS_PATH)
if problemas_comparacion:
    display(Markdown('### Resultados de comparación incompatibles o no disponibles\n\n' + '\n'.join(f'- {problema}' for problema in problemas_comparacion)))
    print('No se fabrican métricas ni gráficos a partir de resultados ausentes o inválidos.')
else:
    print('Se cargó un registro válido de desarrollo y se confirmó que el test no fue evaluado.')

In [ ]:
if comparacion is None:
    print('La tabla comparativa se habilitará cuando exista docs/model_comparison.json válido.')
else:
    tabla = comparacion['tabla'].copy()
    tabla['cobertura_nominal'] = NOMINAL_COVERAGE
    tabla_presentacion = tabla.rename(columns={
        'modelo': 'Modelo',
        'pinball_recommended': 'Pinball p35',
        'mae_recommended': 'MAE p35 (MXN)',
        'coverage': 'Cobertura p10-p90',
        'cobertura_nominal': 'Cobertura nominal',
    })
    display(tabla_presentacion)
    print('Las cifras se muestran tal como fueron cargadas del registro; no se copian manualmente.')

In [ ]:
def graficar_metrica(tabla, columna, titulo, eje_y, referencia=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    tabla.plot(kind='bar', x='modelo', y=columna, ax=ax, legend=False, color='#5b8ff9')
    ax.set_title(titulo)
    ax.set_xlabel('Modelo')
    ax.set_ylabel(eje_y)
    ax.tick_params(axis='x', rotation=0)
    if referencia is not None:
        ax.axhline(referencia, color='#d1495b', linestyle='--', label=f'Nominal: {referencia:.0%}')
        ax.legend()
    fig.tight_layout()
    plt.show()


if comparacion is None:
    print('No se generan gráficos porque faltan métricas reales.')
else:
    graficar_metrica(tabla, 'pinball_recommended', 'Pinball loss del cuantil recomendado p35', 'Pinball loss (MXN)')
    graficar_metrica(tabla, 'mae_recommended', 'MAE del precio recomendado p35', 'MAE (MXN)')
    graficar_metrica(tabla, 'coverage', 'Cobertura observada del intervalo p10-p90', 'Cobertura', referencia=NOMINAL_COVERAGE)

## Análisis por segmentos (opcional)

Esta sección solo se activa con una tabla estructurada real producida por el ejecutor de segmentos. Las dimensiones previstas son alcaldía (`neighbourhood_cleansed`), tipo de alojamiento (`room_type`) y banda de precio (`price_band`). La marca `enough_evidence` debe impedir conclusiones sobre grupos con menos de 30 observaciones.

Si falta `data/processed/segment_tables.csv`, la notebook informa el comando exacto para regenerarlo. `src/segments.py` lee `data/processed/validation_predictions.csv`, une las características de junio y escribe el CSV estructurado y el Markdown; no parseamos el Markdown para reconstruir resultados.

Comando para generar las tablas cuando ya existe la predicción de validación de junio:

```bash
uv run python -m src.segments
```

Si falta `validation_predictions.csv`, primero debe ejecutarse `uv run python -m src.train_models --mode compare` y luego `uv run python -m src.segments`. Ambos comandos pertenecen al flujo de desarrollo con junio; esta notebook no los ejecuta y no evalúa septiembre.

In [ ]:
COMANDO_COMPARACION_JUNIO = 'uv run python -m src.train_models --mode compare'
COMANDO_SEGMENTOS_JUNIO = 'uv run python -m src.segments'

def cargar_segmentos():
    if SEGMENT_CSV_PATH.exists():
        try:
            frame = pd.read_csv(SEGMENT_CSV_PATH)
        except (OSError, UnicodeError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
            return None, f'No se pudo leer la tabla CSV por segmento: {error}.'
        requeridas = {
            'model', 'segment', 'value', 'n', 'pinball_recommended',
            'mae_recommended', 'coverage', 'enough_evidence'
        }
        faltantes = sorted(requeridas - set(frame.columns))
        if faltantes:
            return None, f'La tabla por segmento no tiene las columnas requeridas: {faltantes}.'
        for columna in ['n', 'pinball_recommended', 'mae_recommended', 'coverage']:
            frame[columna] = pd.to_numeric(frame[columna], errors='coerce')
        numericas = frame[['n', 'pinball_recommended', 'mae_recommended', 'coverage']]
        if not np.isfinite(numericas.to_numpy(dtype=float)).all():
            return None, 'La tabla por segmento contiene valores numéricos faltantes o no finitos.'
        if (frame['n'] < 0).any() or (frame['pinball_recommended'] < 0).any() or (frame['mae_recommended'] < 0).any():
            return None, 'La tabla por segmento contiene tamaños o métricas negativas.'
        if ((frame['coverage'] < 0) | (frame['coverage'] > 1)).any():
            return None, 'La tabla por segmento contiene coberturas fuera de [0, 1].'
        return frame, 'CSV estructurado cargado.'
    if PREDICTIONS_PATH.exists():
        comando = COMANDO_SEGMENTOS_JUNIO
    else:
        comando = f'{COMANDO_COMPARACION_JUNIO} y luego {COMANDO_SEGMENTOS_JUNIO}'
    if SEGMENT_MD_PATH.exists():
        return None, (
            'Existe docs/segment_tables.md, pero falta el CSV estructurado. '
            f'Para generarlo de forma reproducible: {comando}. '
            'El Markdown no se parsea automáticamente.'
        )
    return None, f'No hay CSV real de métricas por segmento. Comando necesario: {comando}.'


segmentos, estado_segmentos = cargar_segmentos()
print(estado_segmentos)
if segmentos is None:
    print('No se interpretan alcaldías, tipos de alojamiento ni bandas de precio.')
else:
    segmentos = segmentos.copy()
    segmentos['enough_evidence_bool'] = segmentos['enough_evidence'].astype(str).str.lower().isin({'true', '1', 'yes'})
    columnas = ['model', 'segment', 'value', 'n', 'pinball_recommended', 'mae_recommended', 'coverage', 'enough_evidence']
    display(segmentos[columnas].sort_values(['segment', 'value', 'pinball_recommended']).head(30))

    con_evidencia = segmentos[segmentos['enough_evidence_bool']].copy()
    if con_evidencia.empty:
        print('La tabla no tiene segmentos marcados con evidencia suficiente.')
    else:
        print('Mejor modelo por segmento según pinball p35, calculado desde el CSV cargado:')
        for (tipo_segmento, valor), grupo in con_evidencia.groupby(['segment', 'value'], sort=True):
            mejor = grupo.loc[grupo['pinball_recommended'].idxmin()]
            print(f"- {tipo_segmento} = {valor}: {mejor['model']} (pinball p35 = {mejor['pinball_recommended']:.2f})")

## Interpretación condicionada a los archivos cargados

No se declara ganador por anticipado. La siguiente celda compara el pinball p35 observado en el registro disponible y advierte cuando la cobertura se separa de la referencia nominal. Una mejor métrica de desarrollo no implica automáticamente mejor desempeño en el mercado ni reemplaza la evaluación temporal reservada.

In [ ]:
if comparacion is None:
    print('No hay resultados válidos para interpretar. La notebook queda lista para leerlos después de la integración.')
else:
    ganador = tabla.loc[tabla['pinball_recommended'].idxmin()]
    print(f"Según el archivo cargado, el menor pinball p35 corresponde a {ganador['modelo']}.")
    for _, fila in tabla.iterrows():
        diferencia = fila['coverage'] - NOMINAL_COVERAGE
        sentido = 'por encima' if diferencia >= 0 else 'por debajo'
        print(f"{fila['modelo']}: cobertura {sentido} de la nominal por {abs(diferencia):.1%}.")
    print('Esta comparación no usa septiembre y no convierte la cobertura agregada en una garantía individual.')

## Conclusiones provisionales y limitaciones

Mientras no exista un registro válido de comparación, no hay una conclusión empírica sobre qué familia de modelos es mejor. Cuando se cargue, la conclusión debe limitarse a las particiones de desarrollo de junio y reportar conjuntamente pinball p35, MAE y cobertura.

El precio recomendado representa una referencia basada en precios publicados observados en Inside Airbnb. Puede haber sesgos de selección de anuncios, precios extremos, faltantes, diferencias entre alcaldías y tipos de alojamiento, cambios del mercado y diferencias entre precio publicado y precio finalmente pagado. No debe usarse para garantizar reservas, ocupación, ingresos o rentabilidad.

**Septiembre todavía no fue evaluado en esta notebook.** El test temporal debe permanecer reservado hasta congelar la solución, y no debe usarse para elegir predictores, imputaciones, hiperparámetros, calibración ni modelo.

## Cómo completar la notebook después de la integración

El ejecutor común documentado debe generar, usando únicamente junio:

- `docs/model_comparison.json`: métricas y búsqueda de configuraciones para baseline, Ridge y LightGBM.
- `data/processed/validation_predictions.csv`: predicciones fuera de fold para las tablas por segmento; es regenerable y puede estar ignorado por Git.
- `data/processed/segment_tables.csv`: tabla estructurada opcional para el análisis automático por segmentos.
- `docs/segment_tables.md`: versión legible de las tablas por segmento.

Desde la raíz del repositorio, después de integrar el ejecutor y sus archivos, el protocolo documenta `uv run python -m src.train_models --mode both` y luego `uv run python -m src.segments`. Esos comandos no se ejecutan desde esta notebook. La evaluación final de septiembre queda fuera de este flujo.

## Bitácora de colaboración con IA

- **Validación de datos:** Utilizamos IA para revisar que la separación por `host_id` y el preprocesamiento dentro de cada fold evitaran fugas de información.
- **Integración de modelos:** Utilizamos IA para revisar la compatibilidad entre Ridge, LightGBM, las métricas comunes y la serialización del recomendador.
- **Verificación de resultados:** Utilizamos IA para comprobar que el notebook interpretara correctamente pinball p35, MAE y cobertura utilizando los resultados de desarrollo de junio.